# CEMAT-Stack V4 — Publication Run

Run all cells for the checksum-pinned leakage-safe 5×5 repeated nested-CV pipeline. Completed folds resume from Google Drive.

**The last cell safely handles empty audit files, displays tables/figures inline, and saves the PKL result bundle.**

In [ ]:
!pip -q install -r https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/b115ac27bf953e25e900defd68c03e608a15b06b/requirements-cemat-v4.txt
print('✓ Dependencies installed')

In [ ]:
import os
os.environ['CEMAT_REPEATS'] = '5'
os.environ['CEMAT_BOOTSTRAPS'] = '3000'
os.environ['CEMAT_FORCE_RESTART'] = '0'
print('Scientific configuration: 5 repeats, 5 outer folds, 3000 bootstraps')

In [ ]:
import hashlib
import urllib.request

LOADER_COMMIT = '1c79ed6718d49afeb8589a3eb03c7c7624bb1c30'
EXPECTED_LOADER_SHA256 = 'ddb3e666d6f1ac8fa19e372773ee786cc3e3085c66ac7f175e3f92411eb0b705'
LOADER_URL = ('https://raw.githubusercontent.com/AzizulHakim00/MAT-Appendix/'
              f'{LOADER_COMMIT}/src/v4/verified_loader.py')
loader_bytes = urllib.request.urlopen(LOADER_URL, timeout=120).read()
actual = hashlib.sha256(loader_bytes).hexdigest()
print('Loader commit:', LOADER_COMMIT)
print('Loader SHA256:', actual)
if actual != EXPECTED_LOADER_SHA256:
    raise RuntimeError(f'Loader checksum mismatch: {actual}')
exec(compile(loader_bytes.decode('utf-8'), LOADER_URL, 'exec'), globals(), globals())

In [ ]:
# Self-contained safe result display and PKL export.
from __future__ import annotations
import json, os, pickle, shutil
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from IPython.display import Image, Markdown, display

DRIVE_MY = Path('/content/drive/MyDrive')
if not DRIVE_MY.exists():
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

RUNS_ROOT = Path('/content/drive/MyDrive/MAT-Appendix/cemat_v4_runs')
if not RUNS_ROOT.exists():
    raise RuntimeError(f'CEMAT V4 run directory does not exist: {RUNS_ROOT}')

CRITICAL_FILES = ['summary_mean_std.csv','consensus_metrics.csv','paired_bootstrap.csv','final_decision.json','publication_audit.json','all_repeated_nested_predictions.csv']
def nonempty(path): return path.is_file() and path.stat().st_size > 0
runs = [p for p in RUNS_ROOT.glob('cemat_stack_v4_*') if p.is_dir() and all(nonempty(p/n) for n in CRITICAL_FILES)]
if not runs:
    raise RuntimeError('No completed CEMAT-Stack V4 run was found.')
RUN_DIR = max(runs, key=lambda p: p.stat().st_mtime)
print('='*96)
print('COMPLETED RUN:', RUN_DIR)
print('='*96)

empty_or_skipped, csv_errors, tables = [], {}, {}
def safe_read_csv(path):
    if not path.exists():
        empty_or_skipped.append({'file':path.name,'reason':'missing'}); return None
    if path.stat().st_size == 0:
        empty_or_skipped.append({'file':path.name,'reason':'zero-byte/empty'}); return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except pd.errors.EmptyDataError:
        empty_or_skipped.append({'file':path.name,'reason':'no columns to parse'}); return pd.DataFrame()
    except Exception as exc:
        csv_errors[path.name] = repr(exc)
        empty_or_skipped.append({'file':path.name,'reason':f'read error: {exc}'})
        return None

table_names = ['summary_mean_std.csv','consensus_metrics.csv','repeat_metrics.csv','paired_bootstrap.csv','cohort_summary.csv','leakage_audit.csv','fallback_audit.csv','all_repeated_nested_predictions.csv','per_patient_consensus.csv']
for name in table_names:
    frame = safe_read_csv(RUN_DIR/name)
    if frame is not None: tables[name] = frame

def show_table(title, filename, max_rows=None):
    display(Markdown(f'## {title}'))
    frame = tables.get(filename)
    if frame is None: print(f'{filename}: missing or unreadable')
    elif frame.empty: print(f'{filename}: empty — valid when no audit event occurred.')
    elif max_rows and len(frame)>max_rows:
        display(frame.head(max_rows)); print(f'Showing {max_rows} of {len(frame)} rows; full data is in PKL.')
    else: display(frame)

show_table('Primary repeat-level mean ± SD','summary_mean_std.csv')
show_table('Consensus metrics','consensus_metrics.csv')
show_table('Paired bootstrap comparisons','paired_bootstrap.csv',50)
show_table('Fallback audit','fallback_audit.csv',50)

json_data, json_errors = {}, {}
for name in ['final_decision.json','publication_audit.json','config.json']:
    path = RUN_DIR/name
    if not path.exists() or path.stat().st_size==0:
        json_errors[name]='missing or empty'; continue
    try: json_data[name]=json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc: json_errors[name]=repr(exc)
for name in ['final_decision.json','publication_audit.json']:
    display(Markdown('## '+name.replace('_',' ').replace('.json','').title()))
    print(json.dumps(json_data[name],indent=2) if name in json_data else f'{name}: unavailable ({json_errors.get(name)})')

figure_names=['balanced_metric_comparison.png','high_sensitivity_metric_comparison.png','consensus_roc.png','consensus_pr.png','consensus_calibration.png']
figures={}
for name in figure_names:
    path=RUN_DIR/name
    if path.exists() and path.stat().st_size>0:
        figures[name]=str(path)
        display(Markdown('## '+name.replace('_',' ').replace('.png','').title()))
        display(Image(filename=str(path)))

fold_logs, fold_log_errors = {}, {}
for path in sorted(RUN_DIR.glob('fold_log_repeat*_fold*.json')):
    try: fold_logs[path.name]=json.loads(path.read_text(encoding='utf-8'))
    except Exception as exc: fold_log_errors[path.name]=repr(exc)

bundle={
 'schema':'cemat-stack-v4-complete-results-v2',
 'created_utc':datetime.now(timezone.utc).isoformat(),
 'run_directory':str(RUN_DIR),
 'source_commit':os.environ.get('CEMAT_SOURCE_COMMIT'),
 'source_sha256':os.environ.get('CEMAT_SOURCE_SHA256'),
 'loader_commit':'1c79ed6718d49afeb8589a3eb03c7c7624bb1c30',
 'tables':tables,'json':json_data,'fold_logs':fold_logs,'figures':figures,
 'empty_or_skipped_files':empty_or_skipped,'csv_errors':csv_errors,
 'json_errors':json_errors,'fold_log_errors':fold_log_errors,
 'primary_evidence':'summary_mean_std.csv: repeat-level mean ± SD',
 'notes':{'empty_fallback_audit':'Empty fallback_audit generally means no fallback event; it is not a training failure.','deployment_model':'This PKL is a scientific nested-CV result bundle, not one deployment estimator.'}
}
pkl_path=RUN_DIR/'cemat_stack_v4_complete_results.pkl'
tmp=RUN_DIR/'.cemat_stack_v4_complete_results.pkl.tmp'
with tmp.open('wb') as f: pickle.dump(bundle,f,protocol=pickle.HIGHEST_PROTOCOL)
tmp.replace(pkl_path)
latest=RUNS_ROOT/'LATEST_CEMAT_V4_RESULTS.pkl'
shutil.copy2(pkl_path,latest)
manifest=pd.DataFrame([{'path':str(pkl_path),'bytes':pkl_path.stat().st_size,'status':'saved'},{'path':str(latest),'bytes':latest.stat().st_size,'status':'saved'}])
manifest.to_csv(RUN_DIR/'pkl_export_manifest.csv',index=False)
display(Markdown('## PKL export complete')); display(manifest)
if empty_or_skipped:
    display(Markdown('## Empty or skipped optional files')); display(pd.DataFrame(empty_or_skipped))
print('\n✓ Results displayed successfully.')
print('✓ Run-specific PKL:',pkl_path)
print('✓ Latest PKL:',latest)


## PKL locations

- Run-specific: `MyDrive/MAT-Appendix/cemat_v4_runs/cemat_stack_v4_<hash>/cemat_stack_v4_complete_results.pkl`
- Latest: `MyDrive/MAT-Appendix/cemat_v4_runs/LATEST_CEMAT_V4_RESULTS.pkl`